# Review Analysis & Forecasting
# NeuralProphet VS Prophet
Guest review analysis (6,392 Booking.com reviews, 2023-08-17 to 2026-08-16) covering stay-duration patterns, guest-type breakdowns, and NeuralProphet-based review-score forecasting.

## Setup

In [19]:

!pip install neuralprophet "torch<2.6" plotly-resampler

Defaulting to user installation because normal site-packages is not writeable


In [20]:
import pandas as pd
main_df = pd.read_json("rove_la_mer_reviews_full_v2.jsonl", lines=True)

## Data Overview

In [21]:
main_df.head()

,review_url,username,country,guest_type,guest_review_count,guest_joined_date,room_type,room_type_id,checkin_date,checkout_date,...,title,positive_text,negative_text,lang,reviewed_date,helpful_votes,partner_reply,is_approved,photo_count,photo_urls
0,bf62e74702007c82,Mahmoud,United Arab Emirates,Couple,7,2019-12-21,Rover Room - 36-Hour Staycation with 09:00 AM ...,684556706,2026-07-12,2026-07-13,...,Comfortable,Everything thing is very Good,Nothing,xu,2026-08-16,0.0,None,True,0,None
1,7514f5efaf136d19,Aleksandr,United States,Solo traveler,2,2024-08-21,Rover Room,684556701,2026-07-04,2026-07-12,...,Best value for money,"First of all – stuff. Very welcoming, nice, al...",I would not want to mention anything I didn’t ...,xu,2026-08-16,0.0,None,True,0,None
2,63a7694c04403d69,Raghd,United Arab Emirates,Couple,9,2021-08-15,Rover Room,684556701,2026-08-10,2026-08-12,...,None,None,None,xu,2026-08-16,NaN,None,True,0,None
3,2a210076df471e79,Noor,Oman,Couple,11,2022-01-09,Rover Room,684556701,2026-08-11,2026-08-14,...,None,i liked the vibe at the lobby,the room could be more cozy,en,2026-08-16,0.0,None,True,0,None
4,c1aa43db7320994a,Neeraj,India,Solo traveler,20,2018-06-10,Rover Room,684556701,2026-08-11,2026-08-15,...,None,None,None,en,2026-08-15,NaN,None,True,0,None


In [22]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6392 entries, 0 to 6391
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   review_url          6392 non-null   object 
 1   username            6351 non-null   object 
 2   country             6390 non-null   object 
 3   guest_type          6392 non-null   object 
 4   guest_review_count  6392 non-null   int64  
 5   guest_joined_date   6127 non-null   object 
 6   room_type           6392 non-null   object 
 7   room_type_id        6392 non-null   int64  
 8   checkin_date        6392 non-null   object 
 9   checkout_date       6392 non-null   object 
 10  num_nights          6392 non-null   int64  
 11  customer_type       6392 non-null   object 
 12  stay_status         6392 non-null   object 
 13  review_score        6392 non-null   int64  
 14  title               1834 non-null   object 
 15  positive_text       2944 non-null   object 
 16  negati

In [23]:
print("Date range:", main_df['reviewed_date'].min(), "to", main_df['reviewed_date'].max())
print("Columns:", list(main_df.columns))

Date range: 2023-08-17 to 2026-08-16
Columns: ['review_url', 'username', 'country', 'guest_type', 'guest_review_count', 'guest_joined_date', 'room_type', 'room_type_id', 'checkin_date', 'checkout_date', 'num_nights', 'customer_type', 'stay_status', 'review_score', 'title', 'positive_text', 'negative_text', 'lang', 'reviewed_date', 'helpful_votes', 'partner_reply', 'is_approved', 'photo_count', 'photo_urls']


## Feature Engineering

In [24]:
main_df[["checkin_date", "checkout_date"]] = main_df[["checkin_date", "checkout_date"]].apply(pd.to_datetime)

In [25]:
main_df["stay_duration"] = (main_df["checkout_date"] - main_df["checkin_date"]).dt.days

## Stay Duration Analysis

In [26]:
total_days = main_df['stay_duration'].sum()
print(f"Total cumulative nights stayed across all bookings: {total_days:,}")

Total cumulative nights stayed across all bookings: 21,695


In [27]:
x = main_df.groupby('checkin_date')['stay_duration'].sum().reset_index(name='total_stay_duration').sort_values('total_stay_duration', ascending=False)
y = main_df["stay_duration"].max()

print(x.head(10))
print("\nMaximum stay duration in days:", y, "days")
print(main_df[main_df["stay_duration"] == y][["checkin_date", "checkout_date", "stay_duration"]])

    checkin_date  total_stay_duration
159   2023-12-04                   73
809   2025-09-20                   69
865   2025-11-16                   68
904   2025-12-25                   67
110   2023-10-15                   64
583   2025-02-02                   60
564   2025-01-14                   60
206   2024-01-20                   60
495   2024-11-05                   59
132   2023-11-07                   58

Maximum stay duration in days: 42 days
     checkin_date checkout_date  stay_duration
1340   2025-09-20    2025-11-01             42


## Guest Type Breakdown

In [28]:
df_by_guest_type = main_df.groupby(['reviewed_date', 'guest_type'])['review_score'].agg(['mean', 'sum']).reset_index()
print(df_by_guest_type)

df_by_guest_type_summary = main_df.groupby('guest_type')['review_score'].agg(['mean', 'sum', 'count']).reset_index()
print(df_by_guest_type_summary)

     reviewed_date     guest_type       mean  sum
0       2023-08-17         Couple  10.000000   30
1       2023-08-17         Family   7.000000   28
2       2023-08-17          Group   8.500000   17
3       2023-08-17  Solo traveler  10.000000   10
4       2023-08-18         Couple   9.000000    9
...            ...            ...        ...  ...
2964    2026-08-14         Couple  10.000000   10
2965    2026-08-14         Family   9.000000    9
2966    2026-08-15  Solo traveler  10.000000   10
2967    2026-08-16         Couple   9.666667   29
2968    2026-08-16  Solo traveler  10.000000   10

[2969 rows x 4 columns]
      guest_type      mean    sum  count
0         Couple  8.909313  18273   2051
1         Family  9.010478  21499   2386
2          Group  8.909434   4722    530
3  Solo traveler  8.969123  12781   1425


In [29]:
import sys
!{sys.executable} -m pip install "torchvision==0.20.1"

Defaulting to user installation because normal site-packages is not writeable


## Time Series Forecasting — Overall Trend

In [30]:
daily_df = main_df.groupby('reviewed_date')['review_score'].mean().reset_index()
daily_df.columns = ['ds', 'y']
daily_df['ds'] = pd.to_datetime(daily_df['ds'])

from neuralprophet import NeuralProphet
m = NeuralProphet()
# learning_rate is set explicitly to skip NeuralProphet's automatic
# learning-rate finder -- that step saves/reloads a checkpoint internally,
# which is what triggers the UnpicklingError under PyTorch 2.6+ (see Setup).
metrics = m.fit(daily_df, freq='D', learning_rate=0.1)

future = m.make_future_dataframe(daily_df, periods=30)
forecast = m.predict(future)
forecast['yhat1'] = forecast['yhat1'].clip(1, 10)  # review scores can't exceed 10

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.801% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.config.set_auto_batch_epoch) - Auto-set batch_size to 32
INFO - (NP.config.set_auto_batch_epoch) - Auto-set epochs to 110
WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning:

MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.




Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.801% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 96.667% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 96.667% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 34it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [31]:
import matplotlib.pyplot as plt
import plotly

fig_forecast = m.plot(forecast)
fig_forecast.show()

fig_components = m.plot_components(forecast)
fig_components.show()

fig_model = m.plot_parameters()
fig_model.show()

WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:100: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result




WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:410: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:410: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:410: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future ver

WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/neuralprophet/plot_model_parameters_plotly.py:237: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/neuralprophet/plot_model_parameters_plotly.py:271: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/neuralprophet/plot_model_parameters_plotly.py:475: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is dep

## Time Series Forecasting — By Guest Type

In [32]:
from neuralprophet import NeuralProphet
import pandas as pd

results = {}
forecasts = []

for guest_type in main_df['guest_type'].dropna().unique():
    subset = main_df[main_df['guest_type'] == guest_type]

    daily = subset.groupby('reviewed_date')['review_score'].mean().reset_index()
    daily.columns = ['ds', 'y']
    daily['ds'] = pd.to_datetime(daily['ds'])

    print(f"\n=== {guest_type}: {len(subset)} reviews, {len(daily)} days with data ===")
    if len(daily) < 16:
        print(f"  Skipping -- too little data ({len(daily)} days) for a meaningful fit.")
        continue

    m = NeuralProphet()
    m.fit(daily, freq='D', learning_rate=0.1)

    future = m.make_future_dataframe(daily, periods=10)
    forecast = m.predict(future)
    forecast['guest_type'] = guest_type

    results[guest_type] = m
    forecasts.append(forecast[['ds', 'yhat1', 'guest_type']])

all_forecasts = pd.concat(forecasts, ignore_index=True)
all_forecasts['yhat1'] = all_forecasts['yhat1'].clip(1, 10)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 84.222% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.config.set_auto_batch_epoch) - Auto-set batch_size to 32
INFO - (NP.config.set_auto_batch_epoch) - Auto-set epochs to 120
WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning:

MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.





=== Couple: 2051 reviews, 919 days with data ===


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 84.222% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 90.0% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 90.0% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 29it [00:00, ?it/s]

...


=== Group: 530 reviews, 399 days with data ===


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 41.103% of the data.
WARNING - (NP.df_utils._infer_frequency) - Dataframe has multiple frequencies. It will be resampled according to given freq D. Ignore                     message if actual frequency is any of the following:  SM, BM, CBM, SMS, BMS, CBMS, BQ, BQS, BA,                         or, BAS.
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 90.0% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 90.0% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 25it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [33]:
import plotly.express as px

fig = px.line(all_forecasts, x='ds', y='yhat1', color='guest_type',
              title='Predicted review score, next 10 days, by guest type')
fig.add_hline(y=10, line_dash="dot", line_color="gray")  # visual reminder of the real ceiling
fig.show()

## Model Validation (Backtest)

Train on all but the most recent 7 days, forecast that holdout period, and compare against the real observed values to estimate forecast error.

In [34]:
daily_df = main_df.groupby('reviewed_date')['review_score'].mean().reset_index()
daily_df.columns = ['ds', 'y']
daily_df['ds'] = pd.to_datetime(daily_df['ds'])
daily_df = daily_df.sort_values('ds').reset_index(drop=True)

cutoff = daily_df['ds'].max() - pd.Timedelta(days=7)
train_df = daily_df[daily_df['ds'] <= cutoff]
test_df = daily_df[daily_df['ds'] > cutoff]

print('train rows:', len(train_df), 'test rows:', len(test_df))

from neuralprophet import NeuralProphet
m = NeuralProphet()
m.fit(train_df, freq='D', learning_rate=0.1)

horizon = len(test_df)
future = m.make_future_dataframe(train_df, periods=horizon)
forecast = m.predict(future)
forecast['yhat1'] = forecast['yhat1'].clip(1, 10)

comparison = test_df.merge(forecast[['ds', 'yhat1']], on='ds', how='left')
comparison['abs_error'] = (comparison['y'] - comparison['yhat1']).abs()

print(comparison[['ds', 'y', 'yhat1', 'abs_error']])
print()

mae = comparison['abs_error'].mean()
rmse = (comparison['abs_error'] ** 2).mean() ** 0.5
print(f"MAE  (avg error, in score points): {mae:.2f}")
print(f"RMSE (penalizes bigger misses more): {rmse:.2f}")

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.793% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
INFO - (NP.config.set_auto_batch_epoch) - Auto-set batch_size to 32
INFO - (NP.config.set_auto_batch_epoch) - Auto-set epochs to 110
WARNING - (py.warnings._showwarnmsg) - /Users/elshazlydahab/.local/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning:

MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.




train rows: 1077 test rows: 7


Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.793% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 85.714% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 85.714% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D


Predicting: 34it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


          ds      y     yhat1  abs_error
0 2026-08-10   9.00  9.142505   0.142505
1 2026-08-11   9.00  9.195604   0.195604
2 2026-08-12  10.00  9.166598   0.833402
3 2026-08-13   8.00  9.146167   1.146167
4 2026-08-14   9.50  9.163082   0.336918
5 2026-08-15  10.00  9.213292   0.786708
6 2026-08-16   9.75  9.303388   0.446612

MAE  (avg error, in score points): 0.56
RMSE (penalizes bigger misses more): 0.65


In [35]:
import sys
!{sys.executable} -m pip install prophet

Defaulting to user installation because normal site-packages is not writeable


In [36]:
from prophet import Prophet

# Same daily aggregation as before
daily_df = main_df.groupby('reviewed_date')['review_score'].mean().reset_index()
daily_df.columns = ['ds', 'y']
daily_df['ds'] = pd.to_datetime(daily_df['ds'])
daily_df = daily_df.sort_values('ds').reset_index(drop=True)

# Same 7-day holdout split
cutoff = daily_df['ds'].max() - pd.Timedelta(days=7)
train_df = daily_df[daily_df['ds'] <= cutoff]
test_df = daily_df[daily_df['ds'] > cutoff]

print('train rows:', len(train_df), 'test rows:', len(test_df))

# No learning_rate, no checkpointing, no torch -- Prophet fits via Stan, not gradient descent
m = Prophet()
m.fit(train_df)

# Prophet's make_future_dataframe doesn't take the df again -- it remembers
# its own training data and just extends from there
future = m.make_future_dataframe(periods=len(test_df), freq='D')
forecast = m.predict(future)
forecast['yhat'] = forecast['yhat'].clip(1, 10)  # same score-ceiling clip as before

comparison = test_df.merge(forecast[['ds', 'yhat']], on='ds', how='left')
comparison['abs_error'] = (comparison['y'] - comparison['yhat']).abs()

print(comparison[['ds', 'y', 'yhat', 'abs_error']])
print()

mae = comparison['abs_error'].mean()
rmse = (comparison['abs_error'] ** 2).mean() ** 0.5
print(f"MAE  (avg error, in score points): {mae:.2f}")
print(f"RMSE (penalizes bigger misses more): {rmse:.2f}")

train rows: 1077 test rows: 7


15:43:41 - cmdstanpy - INFO - Chain [1] start processing
15:43:41 - cmdstanpy - INFO - Chain [1] done processing


          ds      y      yhat  abs_error
0 2026-08-10   9.00  8.764975   0.235025
1 2026-08-11   9.00  8.865653   0.134347
2 2026-08-12  10.00  8.833714   1.166286
3 2026-08-13   8.00  8.806206   0.806206
4 2026-08-14   9.50  8.793074   0.706926
5 2026-08-15  10.00  8.921018   1.078982
6 2026-08-16   9.75  8.963031   0.786969

MAE  (avg error, in score points): 0.70
RMSE (penalizes bigger misses more): 0.79
